# Imports

In [ ]:
import torch
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import BaseCallback
import numpy as np

from rl_setup_basic import STEM_environment_history, simulation_parameters, deploy_autofocus

import os

GPU_USE = 3
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # consistent GPU ordering
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_USE)
os.environ["OMP_NUM_THREADS"] = "1"


from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, CallbackList

import os
from datetime import datetime

from tqdm import tqdm

from matplotlib import pyplot as plt
from skimage.metrics import structural_similarity

# simplified haadf simulation or realistic
simulation_parameters["simplified"] = False
simulation_parameters["sampling"] = 0.06

2025-12-05 13:43:40.010469: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-05 13:43:40.020501: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764971020.033209 1824548 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764971020.037364 1824548 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764971020.047878 1824548 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

# Auto focus: stable_baseline3 reinforcement learning

Author: Henry Bell, Updated 11/12/2025

# Training Setup

In [ ]:

# Custom feature extractor 
class CustomCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=256):
        super().__init__(observation_space, features_dim)
        
        n_frames = observation_space.spaces["images"].shape[0]
        h, w = observation_space.spaces["images"].shape[1:]
        n_actions = observation_space.spaces["actions"].shape[0]
        
        self.cnn = nn.Sequential(
            nn.Conv2d(n_frames, 16, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        
        with torch.no_grad():
            dummy = torch.zeros(1, n_frames, h, w)
            cnn_out_size = self.cnn(dummy).shape[1]
        
        self.action_mlp = nn.Sequential(
            nn.Linear(n_actions, 32),
            nn.ReLU(),
        )
        
        self.combined = nn.Sequential(
            nn.Linear(cnn_out_size + 32, features_dim),
            nn.ReLU(),
        )
    
    def forward(self, observations):
        images = observations["images"]
        actions = observations["actions"]
        
        cnn_features = self.cnn(images)
        action_features = self.action_mlp(actions)
        
        combined = torch.cat([cnn_features, action_features], dim=1)
        return self.combined(combined)

check_env(STEM_environment_history(config=simulation_parameters))


# Create model
policy_kwargs = dict(
    features_extractor_class=CustomCNN,
    features_extractor_kwargs=dict(features_dim=256),
)

def make_env():
    return STEM_environment_history(config=simulation_parameters)


/home/hebell/miniconda3/envs/abtem_env/lib/python3.11/site-packages/stable_baselines3/common/env_checker.py:55: UserWarning: It seems that your observation images is an image but its `dtype` is (float32) whereas it has to be `np.uint8`. If your observation is not an image, we recommend you to flatten the observation to have only a 1D vector
  warnings.warn(
/home/hebell/miniconda3/envs/abtem_env/lib/python3.11/site-packages/stable_baselines3/common/env_checker.py:63: UserWarning: It seems that your observation space images is an image but the upper and lower bounds are not in [0, 255]. Because the CNN policy normalize automatically the observation you may encounter issue if the values are not in that range.
  warnings.warn(


# SAC Training Setup

In [ ]:
# create a timestamped base log directory for this run
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
base_log_dir = os.path.join("logs", timestamp)
os.makedirs(base_log_dir, exist_ok=True)
os.makedirs(os.path.join(base_log_dir, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(base_log_dir, "best_sac"), exist_ok=True)
os.makedirs(os.path.join(base_log_dir, "sac_eval"), exist_ok=True)

# Create and normalize env (normalize observations only for off-policy)
def make_env():
    return STEM_environment_history(config=simulation_parameters)

vec_env = DummyVecEnv([lambda: Monitor(make_env())])
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

# Small eval env (unwrapped fresh env, used for deterministic eval)
eval_env = DummyVecEnv([lambda: Monitor(make_env())])
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, clip_obs=10.0)
eval_env.training = False

policy_kwargs = dict(
    features_extractor_class=CustomCNN,
    features_extractor_kwargs=dict(features_dim=256),
)

print("Creating SAC model...")
PREFILL_STEPS = 5000
LEARNING_STARTS = PREFILL_STEPS

model = SAC(
    "MultiInputPolicy",
    vec_env,
    policy_kwargs=policy_kwargs,
    buffer_size=500_000,
    learning_starts=LEARNING_STARTS,
    batch_size=256,        # try 256 or 512
    tau=0.005,
    gamma=0.99,
    train_freq=4,
    gradient_steps=1,
    ent_coef=0.01,   
    learning_rate=3e-4,    # try lowering to 3e-5 if critic unstable
    verbose=1,
    device="cuda",
)
print("SAC model created!\n")

# Prefill replay buffer by stepping the env (let the model collect real transitions)
if PREFILL_STEPS > 0:
    print(f"Prefilling replay buffer with {PREFILL_STEPS} random steps...")
    obs = vec_env.reset()
    steps = 0
    while steps < PREFILL_STEPS:
        actions = [vec_env.action_space.sample() for _ in range(vec_env.num_envs)]
        next_obs, reward, done, info = vec_env.step(actions)
        obs = next_obs
        steps += vec_env.num_envs
    print("Prefill done.")

# Eval callback to monitor deterministic returns

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=os.path.join(base_log_dir, "best_sac"),
    log_path=os.path.join(base_log_dir, "sac_eval"),
    eval_freq=5000,
    n_eval_episodes=5,
    deterministic=True,
)

# Checkpoint callback: save a model every 10k steps
checkpoint_callback = CheckpointCallback(
    save_freq=10_000,
    save_path=os.path.join(base_log_dir, "checkpoints"),
    name_prefix="sac_checkpoint",
    verbose=1,
)

callback = CallbackList([checkpoint_callback, eval_callback])

# Train
model.learn(
    total_timesteps=10000,
    log_interval=10,
    callback=callback,
    progress_bar=True,
) 

# Save model and VecNormalize stats
# model.save("sac_stem_focusing_new_loss_trivial_auto_v2")
vec_env.save(f"{base_log_dir}/vec_logs.pkl")
# ...existing code...

Creating SAC model...
Using cuda device
SAC model created!

Prefilling replay buffer with 5000 random steps...
Prefill done.
Prefill done.


Output()

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 31       |
|    ep_rew_mean     | -6.88    |
| time/              |          |
|    episodes        | 10       |
|    fps             | 3        |
|    time_elapsed    | 79       |
|    total_timesteps | 310      |
---------------------------------


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 31       |
|    ep_rew_mean     | -3.9     |
| time/              |          |
|    episodes        | 20       |
|    fps             | 3        |
|    time_elapsed    | 158      |
|    total_timesteps | 620      |
---------------------------------


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 31       |
|    ep_rew_mean     | -4.73    |
| time/              |          |
|    episodes        | 30       |
|    fps             | 3        |
|    time_elapsed    | 236      |
|    total_timesteps | 930      |
---------------------------------


...

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 31       |
|    ep_rew_mean     | 13       |
| time/              |          |
|    episodes        | 320      |
|    fps             | 3        |
|    time_elapsed    | 2607     |
|    total_timesteps | 9920     |
| train/             |          |
|    actor_loss      | -0.26    |
|    critic_loss     | 0.00192  |
|    ent_coef        | 0.01     |
|    learning_rate   | 0.0003   |
|    n_updates       | 1229     |
---------------------------------


Eval num_timesteps=10000, episode_reward=19.91 +/- 4.50

Episode length: 31.00 +/- 0.00

---------------------------------
| eval/              |          |
|    mean_ep_length  | 31       |
|    mean_reward     | 19.9     |
| time/              |          |
|    total_timesteps | 10000    |
| train/             |          |
|    actor_loss      | -0.252   |
|    critic_loss     | 0.000911 |
|    ent_coef        | 0.01     |
|    learning_rate   | 0.0003   |
|    n_updates       | 1249     |
---------------------------------


New best mean reward!

# Model is trained , let's test it out: We will run 10 episodes and check that it autofocuses them

In [ ]:
env_use = vec_env #vec_env
env_use.training = False

model_use = model

returns = []


def _reset_env(env):
    res = env.reset()
    env.state = 45
    # Gym >=0.26 returns (obs, info); SB3 expects obs only
    if isinstance(res, tuple) and len(res) >= 1:
        return res[0]
    return res

def _step_env(env, action):
    res = env.step(action)
    # Gym >=0.26 step may return (obs, reward, terminated, truncated, info)
    if isinstance(res, tuple) and len(res) == 5:
        obs, reward, terminated, truncated, info = res
        done = bool(terminated or truncated)
        return obs, reward, done, info
    # assume old API: obs, reward, done, info
    return res

states_eps = []
rewards_eps = []
focus_met_eps = []
SSIM_eps = []
fig, ax = plt.subplots(10, 10, figsize = (20, 20))

for j in range(10):
    obs = _reset_env(env_use)
    done = False
    total = 0.0
    states = []
    focus_met = []
    rewards = []
    SSIMs = []
    i = 0
    while not done:
        action, _ = model_use.predict(obs, deterministic=True)
        obs, reward, done, info = _step_env(env_use, action)
        if j < ax.shape[0]:
            if i < ax.shape[1] - 1:
                ax[j, i].imshow(obs["images"][0][-1])
        
                if i ==0:
                    ax[j,i].set_ylabel(f"episode {j}")
                    
                ax[j,i].set_title(f"Timestep {i}")
                
                ax[j,i].set_xticks([])
                ax[j,i].set_yticks([])
        observe = obs["images"][0][0]
              
        potential_array = np.array(info[0]["potential"].array).sum(0)
        potential_array = (potential_array - potential_array.min()) / (potential_array.max() - potential_array.min()) # normalize those bs
        observe = (observe - observe.min()) / (observe.max() - observe.min())
        
        
        
        SSIM = structural_similarity(potential_array, observe, data_range=1.0)
        SSIMs.append(SSIM)
        
        i += 1
        rewards.append(reward)
        states.append(info[0]["state"])
        focus_met.append(info[0]["focus_metric"])
        total += reward
        
    ax[j, -1].imshow(potential_array)
    ax[j,-1].set_xticks([])
    ax[j,-1].set_yticks([])
    ax[j,-1].set_title("Projected potential")
    
    returns.append(total)
    SSIM_eps.append(SSIMs)
    
    rewards_eps.append(rewards)
    states_eps.append(states)
    focus_met_eps.append(focus_met)
print("Deterministic eval mean:", sum(returns)/len(returns))
# env.close()
fig.savefig("images_inference_results_1.svg")

    
# ax[0].set_ylim(-1,1)